# FloodVerify CV: building-level evaluation + resolution ladder (Florence, val)
Runtime: **T4 GPU**. Run top to bottom. About 50-60 min: 15 min data copy, then 4 prediction passes of 8-10 min.

1. **Building level** (the brief: "compute the metrics at the building level where possible"): every labelled building
   counts once, found or missed, with its damage judged by majority vote of its pixels. Gate 2 model and, for the loss
   comparison, the Dice + focal seed-0 model.
2. **Resolution ladder** (Member 3, task 6): the Gate 2 model on Florence images degraded to about 1 m and 3 m
   (factor 2 and 6 from a nominal 0.5 m). No retraining: this measures robustness to coarser imagery.

Validation only. Nepal is scored once, on Thursday, in a separate notebook.

## 0. Setup

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # expect Tesla T4

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [3]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
OUT = f"{D}/eval_extras"
!mkdir -p "{OUT}"
!ls "{CK}"

Mounted at /content/drive
m0_early_fusion_unet_s0  m1_siamese_unet_ce_s2	m1_siamese_unet_s2
m1_siamese_unet_ce_s0	 m1_siamese_unet_s0	smoke_m1_s0
m1_siamese_unet_ce_s1	 m1_siamese_unet_s1


In [4]:
%cd /content
!rm -rf /content/FloodVerify
!git clone -q -b m3/eval-export https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git log --oneline -1
!pip install -q -e . --no-deps jsonschema
import sys; sys.path.insert(0, "/content/FloodVerify/src")

/content
/content/FloodVerify
a1f9090 (HEAD -> m3/eval-export, origin/m3/eval-export) Calibration (temperature, ECE), evaluate CLI, predicted evidence export with crop stitching
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [5]:
# Full data copy (10-15 min). Florence crops are spread over the official train, test and hold folders,
# and the building polygons are in each folder's labels/.
import os
if not os.path.exists(f"{X}/processed/evidence_labels"):
    !mkdir -p /content/xbd/processed
    !tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
    !tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
    !cp -r "{D}/xbd/hold" /content/xbd/
    !cp -r "{D}/xbd/tier3" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
    !unzip -q "{D}/xbd/processed.zip" -d /content/xbd/processed
!ls /content/xbd
!ls /content/xbd/processed/evidence_labels | wc -l   # expect 10148

hold  processed  rscc_flood_manifest.csv  test	tier3  train
10148


## 1. The new code
Not on GitHub yet; this writes it into the installed package. Read the docstring: it defines every building-level number.

In [6]:
%%writefile /content/FloodVerify/src/floodsitrep/cv/eval/building_level.py
"""Building-level evaluation and the resolution ladder. Owner: member 3.

Pixel metrics weigh a large building more than a small one and say nothing about whether a
building was found as a whole. Here every labelled building counts once.

Per 1024 px xBD image (its 512 px crops are predicted, then stitched back together):

1. Ground-truth buildings = the label polygons (Member 1's ``parse_wkt``; ``un-classified``
   skipped), each rasterised to its own pixels. Buildings with no valid pixels are skipped.
2. A building is DETECTED if at least ``detect_share`` (default 0.5) of its valid pixels are
   predicted as building (classes 1-4). Its predicted damage is the majority predicted class
   among those building pixels (ties go to the more severe class). Otherwise it is MISSED.
3. Predicted buildings = 4-connected components of the predicted building mask with at least
   ``min_px`` pixels (the same rule as the evidence export). A component is FALSE if less than
   half of its pixels fall on any labelled building.

Building-level numbers (our definitions, analogous to the xView2 score; state them):
- localisation recall = detected / labelled buildings
- localisation precision = (components - false components) / components
- damage F1 per class over ALL labelled buildings: a missed building is a miss for its class
- building-level xView2-style score = 0.3 x localisation F1 + 0.7 x harmonic mean of the four
  damage F1s.
Confidence intervals: bootstrap over 1024 px images.

Resolution ladder: ``degrade`` = f shrinks both images by f (area averaging) and enlarges them
back to 512 px (bilinear) before the model sees them. f = 2 and 6 approximate 1 m and 3 m from
a nominal 0.5 m GSD. The model is NOT retrained, so this measures robustness of a 0.5 m model
to coarser input, not what a model trained at 3 m could reach.

Needs numpy, scipy, pillow, torch.
"""
import json

import numpy as np

from floodsitrep.cv.eval import metrics as M

CLS_INDEX = {"no-damage": 1, "minor-damage": 2, "major-damage": 3, "destroyed": 4}


def label_buildings(label_path):
    """Label JSON -> list of (class 1-4, polygon points) for classified buildings."""
    from floodsitrep.data.labels_to_evidence import parse_wkt

    with open(label_path) as fh:
        lab = json.load(fh)
    out = []
    for feat in lab["features"]["xy"]:
        k = CLS_INDEX.get(feat["properties"].get("subtype"))
        pts = parse_wkt(feat["wkt"])
        if k is not None and len(pts) >= 3:
            out.append((k, pts))
    return out


def rasterise(buildings, size=1024):
    """Instance map: 0 = no building, i+1 = building i (later polygons draw on top)."""
    from PIL import Image, ImageDraw

    im = Image.new("I", (size, size), 0)
    d = ImageDraw.Draw(im)
    for i, (_, pts) in enumerate(buildings):
        d.polygon(pts, fill=i + 1)
    return np.array(im, dtype=np.int32)


def score_image(pred, valid, buildings, inst=None, detect_share=0.5, min_px=10):
    """One stitched image. pred uint8 [H,W] 0-4, valid bool [H,W].

    Returns (bcm, n_comp, n_false, n_skipped): bcm is a 4x5 matrix, rows = true class 1-4,
    columns = 0 missed, 1-4 predicted class.
    """
    from scipy import ndimage

    if inst is None:
        inst = rasterise(buildings, pred.shape[0])
    bcm = np.zeros((4, 5), np.int64)
    skipped = 0
    pb = (pred > 0) & valid
    nb = len(buildings) + 1
    n_valid = np.bincount(inst[valid].ravel(), minlength=nb)          # valid px per building
    sel = pb & (inst > 0)
    hist = np.bincount((inst[sel] * 5 + pred[sel]).ravel(), minlength=nb * 5).reshape(nb, 5)
    for i, (k, _) in enumerate(buildings):
        n = int(n_valid[i + 1])
        if n == 0:
            skipped += 1
            continue
        counts = hist[i + 1, 1:]                                         # predicted 1-4
        if counts.sum() < detect_share * n:
            bcm[k - 1, 0] += 1
            continue
        bcm[k - 1, 1 + int(np.flatnonzero(counts == counts.max())[-1])] += 1
    lab, nc = ndimage.label(pb)
    n_comp = n_false = 0
    if nc:
        idx = np.arange(1, nc + 1)
        size = ndimage.sum_labels(np.ones_like(lab), lab, idx)
        on_bld = ndimage.sum_labels(inst > 0, lab, idx)
        keep = size >= max(min_px, 1)
        n_comp = int(keep.sum())
        n_false = int((keep & (on_bld < 0.5 * size)).sum())
    return bcm, n_comp, n_false, skipped


def summarise_buildings(bcm, n_comp, n_false):
    """Building-level numbers from a summed 4x5 matrix and component counts."""
    bcm = np.asarray(bcm, dtype=np.float64)
    total = bcm.sum()
    detected = bcm[:, 1:].sum()
    rec = detected / total if total else float("nan")
    prec = (n_comp - n_false) / n_comp if n_comp else float("nan")
    loc = 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0
    f1 = {}
    for c, name in enumerate(("no_damage", "minor", "major", "destroyed")):
        tp = bcm[c, c + 1]
        fn = bcm[c].sum() - tp
        fp = bcm[:, c + 1].sum() - tp
        d = 2 * tp + fp + fn
        f1[name] = float(2 * tp / d) if d else float("nan")
    hm = M.harmonic_mean(list(f1.values()))
    return {"level": "building", "buildings": int(total), "loc_recall": float(rec),
            "loc_precision": float(prec), "loc_f1": float(loc),
            **{f"dmg_f1_{k}": v for k, v in f1.items()}, "damage_f1_hm": hm,
            "xview2_building_score": 0.3 * loc + 0.7 * hm}


def bootstrap_buildings(bcms, comps, falses, n_boot=1000, seed=0):
    """Percentile 95% intervals over images. bcms [N,4,5], comps/falses [N]."""
    bcms, comps, falses = np.asarray(bcms), np.asarray(comps), np.asarray(falses)
    point = summarise_buildings(bcms.sum(0), comps.sum(), falses.sum())
    keys = [k for k, v in point.items() if isinstance(v, float)]
    rng = np.random.default_rng(seed)
    draws = {k: [] for k in keys}
    for _ in range(n_boot):
        i = rng.integers(0, len(bcms), len(bcms))
        s = summarise_buildings(bcms[i].sum(0), comps[i].sum(), falses[i].sum())
        for k in keys:
            draws[k].append(s[k])
    out = {}
    for k in keys:
        a = np.array([x for x in draws[k] if not np.isnan(x)])
        lo, hi = np.quantile(a, [0.025, 0.975]) if len(a) else (float("nan"),) * 2
        out[k] = {"value": point[k], "lo": float(lo), "hi": float(hi)}
    return point, out


def degrade_batch(x, f):
    """Shrink a normalised image batch by factor f (area) and enlarge back (bilinear)."""
    import torch.nn.functional as F

    if f == 1:
        return x
    h, w = x.shape[-2:]
    small = F.interpolate(x, size=(max(1, round(h / f)), max(1, round(w / f))), mode="area")
    return F.interpolate(small, size=(h, w), mode="bilinear", align_corners=False)


def run(checkpoint, xbd_root, manifest, role="val", degrade=1, batch_size=8, limit=0,
        device=None, out_path=None, detect_share=0.5, min_px=10):
    """Predict every crop of ``role``, stitch per image, score pixels and buildings.

    Saves (if out_path) per-crop pixel confusions and per-image building results to .npz.
    """
    import torch
    from torch.utils.data import DataLoader

    from floodsitrep.cv.eval.infer import make_dataset
    from floodsitrep.cv.models import load_checkpoint
    from floodsitrep.data.tiling import PART_ORIGIN, source_paths

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model, _ = load_checkpoint(checkpoint, device)
    ds = make_dataset(xbd_root, manifest, role, limit, sort=True)
    rows = {f"{r['event']}_{r['tile_id']}_p{r['part']}": r for r in ds.rows}
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2,
                    pin_memory=device == "cuda")
    amp = device == "cuda"
    crop_ids, crop_cms = [], []
    img_ids, bcms, comps, falses, skipped = [], [], [], [], []
    state = {"base": None, "pred": None, "valid": None}

    def flush():
        if state["base"] is None:
            return
        row = state["row"]
        blds = label_buildings(source_paths(xbd_root, row)["label"])
        bcm, nc, nf, sk = score_image(state["pred"], state["valid"], blds,
                                      detect_share=detect_share, min_px=min_px)
        img_ids.append(state["base"])
        bcms.append(bcm)
        comps.append(nc)
        falses.append(nf)
        skipped.append(sk)

    model.eval()
    with torch.no_grad():
        for b in dl:
            pre = degrade_batch(b["pre"].to(device), degrade)
            post = degrade_batch(b["post"].to(device), degrade)
            with torch.autocast("cuda", enabled=amp):
                pred = model(pre, post).argmax(1).cpu().numpy().astype(np.uint8)
            masks, valids = b["mask"].numpy(), b["valid"].numpy()
            for i, tid in enumerate(b["tile_id"]):
                crop_ids.append(tid)
                crop_cms.append(M.confusion(pred[i], masks[i], valids[i]))
                row = rows[tid]
                base = tid.rsplit("_p", 1)[0]
                if base != state["base"]:
                    flush()
                    state.update(base=base, row=row,
                                 pred=np.zeros((1024, 1024), np.uint8),
                                 valid=np.zeros((1024, 1024), bool))
                x0, y0 = PART_ORIGIN[row["part"]]
                state["pred"][y0:y0 + 512, x0:x0 + 512] = pred[i]
                state["valid"][y0:y0 + 512, x0:x0 + 512] = valids[i]
    flush()
    res = {"crop_ids": np.array(crop_ids), "crop_cms": np.stack(crop_cms),
           "img_ids": np.array(img_ids), "bcms": np.stack(bcms), "comps": np.array(comps),
           "falses": np.array(falses), "skipped": np.array(skipped)}
    if out_path:
        np.savez_compressed(out_path, **res)
    return res

Writing /content/FloodVerify/src/floodsitrep/cv/eval/building_level.py


In [7]:
import importlib, json, numpy as np, pandas as pd
import floodsitrep.cv.eval.building_level as B
from floodsitrep.cv.eval import metrics as M
importlib.reload(B)
print([f for f in dir(B) if not f.startswith("_")])

['CLS_INDEX', 'M', 'bootstrap_buildings', 'degrade_batch', 'json', 'label_buildings', 'np', 'rasterise', 'run', 'score_image', 'summarise_buildings']


## 2. Prediction passes (8-10 min each; saved runs are skipped, so rerun after a disconnect)

In [8]:
PASSES = [("m1_siamese_unet_ce_s0", 1),   # Gate 2 model, full resolution
          ("m1_siamese_unet_s0", 1),      # Dice + focal, for the building-level loss comparison
          ("m1_siamese_unet_ce_s0", 2),   # about 1 m
          ("m1_siamese_unet_ce_s0", 6)]   # about 3 m
for run, f in PASSES:
    path = f"{OUT}/bl_{run}_val_f{f}.npz"
    if os.path.exists(path):
        print("skip (exists):", path); continue
    print("running", run, "degrade", f, flush=True)
    B.run(f"{CK}/{run}/best.pt", X, MAN, role="val", degrade=f, out_path=path)
R = {(run, f): dict(np.load(f"{OUT}/bl_{run}_val_f{f}.npz")) for run, f in PASSES}
for (run, f), r in R.items():   # sanity: full-resolution pixel scores must match 0.3633 (CE) and 0.3157 (Dice+focal)
    px = M.flat(M.summarise(r["crop_cms"].sum(0)))["xview2_pixel_score"]
    print(f"{run:24s} f={f}  crops {len(r['crop_ids'])}  images {len(r['img_ids'])}  "
          f"buildings {int(r['bcms'].sum())} (+{int(r['skipped'].sum())} with no valid pixels)  pixel score {px:.4f}")

skip (exists): /content/drive/MyDrive/MAAI 902 X 901 PROJECT/eval_extras/bl_m1_siamese_unet_ce_s0_val_f1.npz
skip (exists): /content/drive/MyDrive/MAAI 902 X 901 PROJECT/eval_extras/bl_m1_siamese_unet_s0_val_f1.npz
skip (exists): /content/drive/MyDrive/MAAI 902 X 901 PROJECT/eval_extras/bl_m1_siamese_unet_ce_s0_val_f2.npz
skip (exists): /content/drive/MyDrive/MAAI 902 X 901 PROJECT/eval_extras/bl_m1_siamese_unet_ce_s0_val_f6.npz
m1_siamese_unet_ce_s0    f=1  crops 2184  images 546  buildings 10728 (+0 with no valid pixels)  pixel score 0.3633
m1_siamese_unet_s0       f=1  crops 2184  images 546  buildings 10728 (+0 with no valid pixels)  pixel score 0.3157
m1_siamese_unet_ce_s0    f=2  crops 2184  images 546  buildings 10728 (+0 with no valid pixels)  pixel score 0.3216
m1_siamese_unet_ce_s0    f=6  crops 2184  images 546  buildings 10728 (+0 with no valid pixels)  pixel score 0.0251


## 3. Building level vs pixel level, with 95% bootstrap intervals over images

In [9]:
def table(run, f):
    r = R[(run, f)]
    point, ci = B.bootstrap_buildings(r["bcms"], r["comps"], r["falses"])
    px = M.flat(M.summarise(r["crop_cms"].sum(0)))
    rows = []
    for k_b, k_p in [("xview2_building_score", "xview2_pixel_score"), ("loc_f1", "loc_f1"),
                     ("loc_recall", None), ("loc_precision", None),
                     ("dmg_f1_no_damage", "dmg_f1_no_damage"), ("dmg_f1_minor", "dmg_f1_minor"),
                     ("dmg_f1_major", "dmg_f1_major"), ("dmg_f1_destroyed", "dmg_f1_destroyed"),
                     ("damage_f1_hm", "damage_f1_hm")]:
        c = ci[k_b]
        rows.append({"metric": k_b, "building": c["value"], "95% lo": c["lo"], "95% hi": c["hi"],
                     "pixel": px[k_p] if k_p else np.nan})
    return pd.DataFrame(rows).round(4)
for run in ("m1_siamese_unet_ce_s0", "m1_siamese_unet_s0"):
    print("\n", run, "(full resolution)")
    print(table(run, 1).to_string(index=False))


 m1_siamese_unet_ce_s0 (full resolution)
               metric  building  95% lo  95% hi  pixel
xview2_building_score    0.3193  0.2818  0.3472 0.3633
               loc_f1    0.7783  0.7559  0.7998 0.7820
           loc_recall    0.8578  0.8455  0.8697    NaN
        loc_precision    0.7123  0.6771  0.7478    NaN
     dmg_f1_no_damage    0.6929  0.6403  0.7409 0.6675
         dmg_f1_minor    0.0578  0.0362  0.0847 0.1002
         dmg_f1_major    0.4392  0.3647  0.5151 0.4492
     dmg_f1_destroyed    0.0863  0.0314  0.1460 0.1240
         damage_f1_hm    0.1226  0.0703  0.1608 0.1838

 m1_siamese_unet_s0 (full resolution)
               metric  building  95% lo  95% hi  pixel
xview2_building_score    0.2864  0.2318  0.3168 0.3157
               loc_f1    0.7729  0.7552  0.7909 0.7935
           loc_recall    0.7403  0.7228  0.7579    NaN
        loc_precision    0.8086  0.7724  0.8401    NaN
     dmg_f1_no_damage    0.5960  0.5499  0.6408 0.6191
         dmg_f1_minor    0.0507  0.0327

In [10]:
# Where do labelled buildings end up? Rows: true class; columns: missed or predicted class (row %)
r = R[("m1_siamese_unet_ce_s0", 1)]
bcm = r["bcms"].sum(0)
names = ["no damage", "minor", "major", "destroyed"]
df = pd.DataFrame(bcm, index=["true " + n for n in names], columns=["missed"] + ["pred " + n for n in names])
print(df)
print((df.div(df.sum(1), axis=0) * 100).round(1))
print("false building components:", int(r["falses"].sum()), "of", int(r["comps"].sum()))

                missed  pred no damage  pred minor  pred major  pred destroyed
true no damage    1214            4602        2085         479              86
true minor          33              67          92          37               3
true major         251             145         758         699              96
true destroyed      27               4          19          19              12
                missed  pred no damage  pred minor  pred major  pred destroyed
true no damage    14.3            54.4        24.6         5.7             1.0
true minor        14.2            28.9        39.7        15.9             1.3
true major        12.9             7.4        38.9        35.9             4.9
true destroyed    33.3             4.9        23.5        23.5            14.8
false building components: 3309 of 11502


## 4. Resolution ladder (Gate 2 model, no retraining)

In [11]:
rows = []
for f, label in [(1, "0.5 m (original)"), (2, "about 1 m"), (6, "about 3 m")]:
    r = R[("m1_siamese_unet_ce_s0", f)]
    b = B.summarise_buildings(r["bcms"].sum(0), r["comps"].sum(), r["falses"].sum())
    p = M.flat(M.summarise(r["crop_cms"].sum(0)))
    rows.append({"resolution": label, "pixel score": p["xview2_pixel_score"], "pixel loc F1": p["loc_f1"],
                 "building score": b["xview2_building_score"], "building recall": b["loc_recall"],
                 "building precision": b["loc_precision"], "bld F1 major": b["dmg_f1_major"],
                 "bld F1 destroyed": b["dmg_f1_destroyed"]})
pd.DataFrame(rows).round(4)

,resolution,pixel score,pixel loc F1,building score,building recall,building precision,bld F1 major,bld F1 destroyed
0,0.5 m (original),0.3633,0.7820,0.3193,0.8578,0.7123,0.4392,0.0863
1,about 1 m,0.3216,0.7341,0.3049,0.7264,0.7804,0.2323,0.0797
2,about 3 m,0.0251,0.0502,0.0312,0.0368,0.7936,0.0072,0.0143


## Done: paste the outputs of sections 2, 3 and 4 in the chat, then stop billing

In [ ]:
from google.colab import runtime
runtime.unassign()